# Notebook 27 — Online Phase-Lock Control and Hardware Mapping

Closed-loop control with CGCS constraint gating applied **before** simulated hardware updates.

Repository path:

`control-stack/27_online_phase_lock_control_and_hardware_mapping.ipynb`

Core idea:

> certification must happen before actuation, not after evaluation.

In [ ]:
# ============================================================
# Imports, constants, and output paths
# ============================================================
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "27"
SLUG = "online_phase_lock_control_and_hardware_mapping"
TITLE = "Online Phase-Lock Control and Hardware Mapping"
SEED = 9423
THRESHOLD = 1 / np.sqrt(1**2 + 1**2)  # 45° gate

np.random.seed(SEED)

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
DOCS_DIR = ROOT / "docs"
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Notebook {NOTEBOOK_ID}: {TITLE}")
print(f"45° cosine threshold = {THRESHOLD:.6f}")

## 1. Online control loop

Notebook 26 packaged the certification figures. Notebook 27 turns the same idea into a runtime loop:

```text
measurement → estimate → policy → CGCS gate → hardware update → next measurement
```

The CGCS gate is active here. A proposed update is accepted, damped, or rejected before it reaches the simulated hardware state.

In [ ]:
# ============================================================
# Synthetic online hardware drift model
# ============================================================
def build_reference_system(T=140, seed=9423):
    """Create true hardware drift, measurements, and stress windows."""
    rng = np.random.default_rng(seed)
    t = np.arange(T)

    # Slowly evolving hardware drift parameters.
    true_omega = 0.035*np.sin(2*np.pi*t/38) + 0.018*np.sin(2*np.pi*t/13)
    true_beta = 0.010 + 0.00045*t + 0.012*np.sin(2*np.pi*t/57)

    # Structured perturbation windows.
    stress_windows = [
        (30, 45, "noise burst"),
        (58, 72, "model mismatch"),
        (88, 104, "noise burst"),
        (112, 128, "hardware drift ramp"),
    ]
    disturbance = np.zeros(T)
    beta_disturbance = np.zeros(T)

    for start, stop, label in stress_windows:
        width = stop - start
        local = np.linspace(-1, 1, width)
        if "noise" in label:
            disturbance[start:stop] += 0.060*np.exp(-4*local**2) * np.sign(np.sin(np.linspace(0, np.pi, width)))
            beta_disturbance[start:stop] += 0.028*np.exp(-5*local**2)
        elif "mismatch" in label:
            disturbance[start:stop] += -0.045*np.tanh(3*local)
            beta_disturbance[start:stop] += 0.020*np.cos(np.linspace(0, np.pi, width))
        else:
            disturbance[start:stop] += np.linspace(0.0, 0.055, width)
            beta_disturbance[start:stop] += np.linspace(0.0, -0.025, width)

    true_omega = true_omega + disturbance
    true_beta = true_beta + beta_disturbance

    measured_omega = true_omega + rng.normal(0, 0.013, T)
    measured_beta = true_beta + rng.normal(0, 0.010, T)

    # Make a few bad measurements to motivate gating.
    for idx, amp in [(22, 0.08), (66, -0.10), (96, 0.09), (121, -0.07)]:
        measured_omega[idx] += amp
        measured_beta[idx] -= 0.5*amp

    return {
        "t": t,
        "true_omega": true_omega,
        "true_beta": true_beta,
        "measured_omega": measured_omega,
        "measured_beta": measured_beta,
        "stress_windows": stress_windows,
    }

system = build_reference_system()
print("system keys:", list(system.keys()))
print("T =", len(system["t"]))

## 2. Controller policies

Each policy proposes a hardware update vector:

```text
u_t = [Ω update, B update]
```

The CGCS variant differs because it applies a 45° stability gate before actuation.

In [ ]:
# ============================================================
# Estimators and control policies
# ============================================================
def moving_average_estimate(history, window=7):
    if len(history) == 0:
        return np.zeros(2)
    arr = np.array(history[-window:])
    return arr.mean(axis=0)

class SimpleKalman2D:
    def __init__(self, q=0.0015, r=0.012):
        self.x = np.zeros(2)
        self.P = np.eye(2)*0.04
        self.Q = np.eye(2)*q
        self.R = np.eye(2)*r

    def step(self, z):
        # random-walk prediction
        self.P = self.P + self.Q
        S = self.P + self.R
        K = self.P @ np.linalg.inv(S)
        self.x = self.x + K @ (z - self.x)
        self.P = (np.eye(2) - K) @ self.P
        return self.x.copy()


def propose_update(policy, z, estimate, target, prev_u, history):
    """Return proposed update vector [omega, beta]."""
    if policy == "none":
        return z.copy()
    if policy == "moving_average":
        return moving_average_estimate(history + [z], window=9)
    if policy == "joint_kalman":
        return estimate.copy()
    if policy == "robust_gated_kalman":
        # Conservative blend toward filtered estimate.
        return 0.70*prev_u + 0.30*estimate
    if policy == "constrained_mpc":
        # Smooth actuation: prefer small moves, correct toward current measurement.
        raw = 0.55*estimate + 0.25*z + 0.20*target
        max_step = 0.022
        delta = np.clip(raw - prev_u, -max_step, max_step)
        return prev_u + delta
    if policy == "cgcs_invariance_control":
        # Aggressive but later gated by CGCS.
        raw = 0.62*estimate + 0.28*z + 0.10*target
        max_step = 0.035
        delta = np.clip(raw - prev_u, -max_step, max_step)
        return prev_u + delta
    if policy == "oracle":
        return target.copy()
    raise ValueError(f"unknown policy: {policy}")


def waveform_from_params(params, tau):
    """Toy excited-state probability from hardware update parameters."""
    omega, beta = params
    phase = (1.0 + 2.4*omega) * tau + 3.0*beta
    contrast = np.clip(0.50 - 1.4*abs(beta), 0.32, 0.52)
    offset = 0.50 + 0.25*beta
    return np.clip(offset + contrast*np.sin(phase)**2, 0, 1.05)


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a)*np.linalg.norm(b)
    if denom == 0:
        return 1.0
    return float(np.dot(a, b) / denom)


def cgcs_margin(candidate, target, prev_u, safety_anchor=None):
    """Compute phase-lock margin above 45° threshold.

    Compare candidate direction against both target direction and update continuity.
    """
    if safety_anchor is None:
        safety_anchor = 0.65*target + 0.35*prev_u
    cos_target = cosine_similarity(candidate + 1e-6, safety_anchor + 1e-6)
    cos_step = cosine_similarity(candidate - prev_u + 1e-6, target - prev_u + 1e-6)
    cos_score = min(cos_target, cos_step)
    return cos_score - THRESHOLD, cos_score

POLICIES = [
    "none",
    "moving_average",
    "joint_kalman",
    "robust_gated_kalman",
    "constrained_mpc",
    "cgcs_invariance_control",
    "oracle",
]

print(POLICIES)

## 3. Runtime CGCS gate

The gate checks whether a proposed hardware update remains inside the 45° phase-lock cone. If not, the update is damped or rejected.

In [ ]:
# ============================================================
# Online control simulation
# ============================================================
def run_online_controller(policy, system, gate_mode="dampen", seed=9423):
    rng = np.random.default_rng(seed)
    t = system["t"]
    T = len(t)
    kalman = SimpleKalman2D(q=0.0008 if "kalman" in policy else 0.0015, r=0.010)

    prev_u = np.array([0.0, 0.0])
    target_history = []
    measurement_history = []
    rows = []

    tau = np.linspace(0, 10, 360)

    for k in range(T):
        true_state = np.array([system["true_omega"][k], system["true_beta"][k]])
        z = np.array([system["measured_omega"][k], system["measured_beta"][k]])
        measurement_history.append(z)

        # Hardware target is the true drift compensation direction.
        target = true_state.copy()
        target_history.append(target)

        estimate = kalman.step(z)
        proposed = propose_update(policy, z, estimate, target, prev_u, measurement_history)

        margin, cos_score = cgcs_margin(proposed, target, prev_u)
        accepted = True
        applied = proposed.copy()
        gate_action = "apply"

        if policy == "cgcs_invariance_control":
            if margin < 0:
                accepted = False
                if gate_mode == "reject":
                    applied = prev_u.copy()
                    gate_action = "reject"
                elif gate_mode == "dampen":
                    # scale back toward previous safe update until the margin is nonnegative
                    applied = prev_u.copy()
                    gate_action = "dampen"
                    for alpha in np.linspace(0.85, 0.05, 17):
                        trial = prev_u + alpha*(proposed - prev_u)
                        m_trial, c_trial = cgcs_margin(trial, target, prev_u)
                        if m_trial >= 0:
                            applied = trial
                            margin, cos_score = m_trial, c_trial
                            accepted = True
                            gate_action = "dampen_accept"
                            break
                else:
                    raise ValueError("gate_mode must be reject or dampen")

        # Non-CGCS policies are not actively gated, but we still evaluate margin.
        if policy != "cgcs_invariance_control":
            accepted = margin >= 0
            gate_action = "diagnostic_only"

        y_target = waveform_from_params(target, tau)
        y_applied = waveform_from_params(applied, tau)
        rmse = float(np.sqrt(np.mean((y_applied - y_target)**2)))

        rows.append({
            "block": int(k),
            "policy": policy,
            "true_omega": true_state[0],
            "true_beta": true_state[1],
            "measured_omega": z[0],
            "measured_beta": z[1],
            "estimate_omega": estimate[0],
            "estimate_beta": estimate[1],
            "proposed_omega": proposed[0],
            "proposed_beta": proposed[1],
            "applied_omega": applied[0],
            "applied_beta": applied[1],
            "margin": margin,
            "cos_score": cos_score,
            "accepted": bool(accepted),
            "gate_action": gate_action,
            "rmse": rmse,
        })

        prev_u = applied.copy()

    return pd.DataFrame(rows)

logs = pd.concat([run_online_controller(p, system) for p in POLICIES], ignore_index=True)
logs.head()

In [ ]:
# ============================================================
# Summary metrics
# ============================================================
def recovery_time_after_failure(df, max_steps=12):
    """Mean blocks needed to return to margin >= 0 after a negative-margin event."""
    times = []
    margins = df["margin"].to_numpy()
    for i, m in enumerate(margins):
        if m < 0:
            recovered = None
            for j in range(i+1, min(len(margins), i+max_steps+1)):
                if margins[j] >= 0:
                    recovered = j - i
                    break
            times.append(max_steps if recovered is None else recovered)
    return 0.0 if len(times) == 0 else float(np.mean(times))

summary_rows = []
for policy, df in logs.groupby("policy"):
    summary_rows.append({
        "policy": policy,
        "mean_rmse": df["rmse"].mean(),
        "p95_rmse": df["rmse"].quantile(0.95),
        "max_rmse": df["rmse"].max(),
        "min_margin": df["margin"].min(),
        "mean_margin": df["margin"].mean(),
        "blocks_below_threshold": int((df["margin"] < 0).sum()),
        "accept_rate": float((df["margin"] >= 0).mean()),
        "active_gate_rejections": int((df["gate_action"] == "reject").sum()),
        "active_gate_dampens": int(df["gate_action"].isin(["dampen", "dampen_accept"]).sum()),
        "mean_recovery_time": recovery_time_after_failure(df),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df["certification_score"] = (
    1.2*(summary_df["min_margin"].clip(lower=-0.35) + 0.35)
    + 0.7*(1 - summary_df["mean_rmse"] / (summary_df["mean_rmse"].max() + 1e-9))
    + 0.4*(1 - summary_df["blocks_below_threshold"] / (summary_df["blocks_below_threshold"].max() + 1e-9))
)
summary_df = summary_df.sort_values("certification_score", ascending=False).reset_index(drop=True)
summary_df

## 4. Figures

Notebook 27 focuses on runtime behavior:

1. online Ω trace
2. online B trace
3. CGCS margin trace
4. accepted / damped / rejected gate actions
5. RMSE over time
6. certification summary
7. hardware mapping diagram
8. worst-case waveform comparison

In [ ]:
# ============================================================
# Plot helpers
# ============================================================
def shade_stress_windows(ax, stress_windows):
    for start, stop, label in stress_windows:
        alpha = 0.16 if "noise" in label else 0.10
        ax.axvspan(start, stop, alpha=alpha, label=label if start == stress_windows[0][0] else None)

def savefig(name):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"saved figure: {path}")

ordered = summary_df["policy"].tolist()
ordered

In [ ]:
# Figure 1: Ω tracking
plt.figure(figsize=(13, 5))
ax = plt.gca()
shade_stress_windows(ax, system["stress_windows"])
plt.plot(system["t"], system["true_omega"], lw=2.5, label="true Ω drift")
plt.plot(system["t"], system["measured_omega"], marker="o", ms=3, lw=1.0, alpha=0.65, label="measured Ω")
for p in ordered:
    if p == "oracle":
        continue
    df = logs[logs.policy == p]
    plt.plot(df["block"], df["applied_omega"], lw=1.8, label=p)
plt.title("Online phase-lock control: Ω hardware-update trace")
plt.xlabel("calibration block")
plt.ylabel("Ω estimate / applied command")
plt.legend(ncol=3, fontsize=8)
savefig("omega_tracking")

In [ ]:
# Figure 2: B tracking
plt.figure(figsize=(13, 5))
ax = plt.gca()
shade_stress_windows(ax, system["stress_windows"])
plt.plot(system["t"], system["true_beta"], lw=2.5, label="true B drift")
plt.plot(system["t"], system["measured_beta"], marker="o", ms=3, lw=1.0, alpha=0.65, label="measured B")
for p in ordered:
    if p == "oracle":
        continue
    df = logs[logs.policy == p]
    plt.plot(df["block"], df["applied_beta"], lw=1.8, label=p)
plt.title("Online phase-lock control: B hardware-update trace")
plt.xlabel("calibration block")
plt.ylabel("B estimate / applied command")
plt.legend(ncol=3, fontsize=8)
savefig("beta_tracking")

In [ ]:
# Figure 3: CGCS margin trace
plt.figure(figsize=(13, 5))
ax = plt.gca()
shade_stress_windows(ax, system["stress_windows"])
for p in ordered:
    df = logs[logs.policy == p]
    plt.plot(df["block"], df["margin"], marker="o", ms=3, lw=1.4, label=p)
plt.axhline(0, ls="--", lw=1.5, label="threshold margin = 0")
plt.title("Online phase-lock control: runtime CGCS margin")
plt.xlabel("calibration block")
plt.ylabel("cosine margin above 45° threshold")
plt.legend(ncol=3, fontsize=8)
savefig("runtime_margin")

In [ ]:
# Figure 4: CGCS gate actions
cgcs_df = logs[logs.policy == "cgcs_invariance_control"].copy()
plt.figure(figsize=(12, 4.5))
plt.plot(cgcs_df["block"], cgcs_df["margin"], lw=2, label="CGCS margin")
plt.axhline(0, ls="--", label="threshold margin = 0")
for action, marker in [("dampen_accept", "o"), ("dampen", "x"), ("reject", "s")]:
    sub = cgcs_df[cgcs_df["gate_action"] == action]
    if len(sub):
        plt.scatter(sub["block"], sub["margin"], s=70, marker=marker, label=action)
plt.title("Online CGCS gate actions before hardware update")
plt.xlabel("calibration block")
plt.ylabel("margin")
plt.legend()
savefig("cgcs_gate_actions")

In [ ]:
# Figure 5: response RMSE over time
plt.figure(figsize=(13, 5))
ax = plt.gca()
shade_stress_windows(ax, system["stress_windows"])
for p in ordered:
    df = logs[logs.policy == p]
    plt.plot(df["block"], df["rmse"], marker="o", ms=3, lw=1.4, label=p)
plt.title("Online phase-lock control: response RMSE over time")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target waveform")
plt.legend(ncol=3, fontsize=8)
savefig("rmse_over_time")

In [ ]:
# Figure 6: certification scorecard
plt.figure(figsize=(10, 5.5))
plt.barh(summary_df["policy"], summary_df["certification_score"])
plt.gca().invert_yaxis()
plt.title("Online control certification scorecard")
plt.xlabel("certification score")
for i, row in summary_df.iterrows():
    label = "certified" if (row.min_margin >= 0 and row.blocks_below_threshold == 0) else "runtime risk"
    plt.text(row.certification_score + 0.02, i, label, va="center")
savefig("online_certification_scorecard")

In [ ]:
# Figure 7: hardware mapping diagram
plt.figure(figsize=(13, 5))
ax = plt.gca()
ax.axis("off")
boxes = [
    (0.05, 0.58, "Calibration\nmeasurements"),
    (0.25, 0.58, "Drift/noise\nestimation"),
    (0.45, 0.58, "Controller\npolicy"),
    (0.65, 0.58, "CGCS 45°\nconstraint gate"),
    (0.85, 0.58, "Hardware\nupdate"),
]
for x, y, text in boxes:
    rect = plt.Rectangle((x, y), 0.13, 0.18, fill=False, lw=2)
    ax.add_patch(rect)
    ax.text(x+0.065, y+0.09, text, ha="center", va="center", fontsize=12)
for x in [0.18, 0.38, 0.58, 0.78]:
    ax.annotate("", xy=(x+0.06, 0.67), xytext=(x, 0.67), arrowprops=dict(arrowstyle="->", lw=2))
ax.text(0.5, 0.34, r"runtime condition:  $\cos\theta \geq 1/\sqrt{1^2+1^2}$", ha="center", fontsize=16)
ax.text(0.5, 0.22, "RMSE measures accuracy; CGCS margin measures stability before actuation.", ha="center", fontsize=12)
plt.title("Online control-stack architecture: measurement → policy → CGCS gate → hardware update", fontsize=16)
savefig("hardware_mapping_diagram")

In [ ]:
# Figure 8: worst-case waveform comparison
worst_block = int(logs[logs.policy != "oracle"].sort_values("rmse", ascending=False).iloc[0]["block"])
tau = np.linspace(0, 10, 360)
plt.figure(figsize=(12, 5))
true_params = np.array([system["true_omega"][worst_block], system["true_beta"][worst_block]])
plt.plot(tau, waveform_from_params(true_params, tau), lw=3, label="target")
for p in ordered:
    df = logs[(logs.policy == p) & (logs.block == worst_block)].iloc[0]
    params = np.array([df.applied_omega, df.applied_beta])
    plt.plot(tau, waveform_from_params(params, tau), lw=1.8, label=p)
plt.title(f"Online control: worst-case waveform comparison — block {worst_block}")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=3, fontsize=8)
savefig("worst_case_waveform")

## 5. Hardware interpretation

| Abstract notebook variable | Hardware interpretation |
|---|---|
| `Ω` | Rabi-frequency / pulse-amplitude correction |
| `B` | bias-field / detuning-related correction |
| `u_t` | proposed hardware update vector |
| `margin` | CGCS stability margin above 45° threshold |
| `gate_action` | apply, dampen, or reject update |
| `rmse` | accuracy relative to target response |

The main claim is not that CGCS has lower RMSE in every local window. The claim is stronger and more hardware-oriented:

> CGCS creates a runtime safety gate that prevents unstable updates before they are applied.

In [ ]:
# ============================================================
# Tables, markdown report, manifest, and export zip
# ============================================================
figure_names = [
    "omega_tracking",
    "beta_tracking",
    "runtime_margin",
    "cgcs_gate_actions",
    "rmse_over_time",
    "online_certification_scorecard",
    "hardware_mapping_diagram",
    "worst_case_waveform",
]

logs_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_online_logs.csv"
summary_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_summary.csv"
logs.to_csv(logs_path, index=False)
summary_df.to_csv(summary_path, index=False)

best_policy = summary_df.iloc[0]["policy"]
best_cgcs_policy = "cgcs_invariance_control"

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}\n")
md_lines.append("## Summary\n")
md_lines.append(
    "Notebook 27 implements an online control loop where proposed hardware updates are evaluated "
    "by a CGCS 45° constraint gate before actuation. It converts the previous certification stack "
    "from offline evaluation into runtime control.\n"
)
md_lines.append("## Key result\n")
md_lines.append(
    f"- Best policy by certification score: `{best_policy}`\n"
    f"- Active CGCS policy: `{best_cgcs_policy}`\n"
    f"- Threshold: `cos(theta) >= {THRESHOLD:.6f}`\n"
)
md_lines.append("## Online certification table\n")
md_lines.append(summary_df.to_markdown(index=False))
md_lines.append("\n## Hardware mapping\n")
md_lines.append(
    "`Ω` maps to pulse-amplitude / Rabi-frequency corrections; `B` maps to bias-field or detuning-related corrections. "
    "The CGCS margin is interpreted as a stability gate, while RMSE measures target-response accuracy.\n"
)
md_lines.append("## Figures\n")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)\n")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("\n".join(md_lines))
print(f"saved markdown: {md_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "seed": SEED,
    "threshold": THRESHOLD,
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [str(logs_path), str(summary_path)],
    "docs": [str(md_path)],
    "core_claim": "CGCS gates proposed hardware updates before actuation, reducing unstable runtime decisions.",
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")

zip_path = ROOT / f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in manifest["figures"] + manifest["results"] + manifest["docs"]:
        p = Path(path)
        if p.exists():
            zf.write(p, arcname=str(p))
print(f"saved zip: {zip_path}")

## 6. Optional Colab download

Run this final cell in Colab to download the generated output bundle.

In [ ]:
# Optional Colab download
try:
    from google.colab import files
    files.download(f"{NOTEBOOK_ID}_{SLUG}_outputs.zip")
except Exception as exc:
    print("Colab download skipped:", exc)

## README snippet

```markdown
### Notebook 27 — Online Phase-Lock Control and Hardware Mapping

Implements a real-time control loop with CGCS constraint gating applied before hardware updates.

Demonstrates:
- online measurement → estimation → policy → gate → update flow
- runtime margin checks against the 45° threshold
- damped CGCS updates when proposed actuation leaves the stable cone
- hardware interpretation for Ω and B control primitives

Core idea: certification must happen before actuation, not after evaluation.
```